# Session 3 — Prompt Engineering for Biomedical Abstracts

*Statistical Foundations of LLMs · Session 3 · Interactive activity (accompanies slide 260)*

The same model can produce a brilliant answer or a useless one depending entirely on **how you ask**. In this activity you will write, refine, and compare prompts that ask a language model to draft and summarize **biomedical abstracts** — then evaluate the results with both your judgment and simple automatic metrics.

We use **open models** (BioGPT and FLAN-T5) so everything runs in Colab with **no API key**. Optional GPT-4 cells are clearly marked for anyone who has a key.

> **EXECUTED SOLUTIONS NOTEBOOK.** Pre-run instructor copy with all code executed and outputs saved. Source: `notebooks/solutions/`. Regenerate with `python scripts/export_executed_solutions.py`.


> ✅ **SOLUTIONS NOTEBOOK.** Every exercise stub is filled in and every challenge cell contains working reference code. Use this for self-check or as an instructor key — encourage students to attempt the exercises in the main notebook first.

## Learning Objectives

1. Explain what prompt engineering is and why prompts steer model behavior.
2. Apply **zero-shot, one-shot, and few-shot** prompting to biomedical text.
3. Practice **iterative prompt refinement** — improve a prompt across several rounds.
4. **Compare** competing prompts systematically on the same task.
5. **Evaluate** generated abstracts with a rubric plus ROUGE overlap.


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background: What Is Prompt Engineering?

A language model computes $P(\text{next tokens} \mid \text{your prompt})$. The prompt is the *only* lever you have at inference time — you are not changing the weights, you are **conditioning** the distribution. Good prompts:

- state the **task** explicitly ("Summarize the following abstract in two sentences for a non-specialist");
- supply **structure** (role, input, constraints, output format);
- optionally include **examples** (one-shot / few-shot) that demonstrate the desired behavior — this is **in-context learning**.

| Style | What you give the model | When to use |
|---|---|---|
| **Zero-shot** | Instruction only | Simple, common tasks |
| **One-shot** | Instruction + 1 example | Show a format |
| **Few-shot** | Instruction + several examples | Nuanced style/format, harder tasks |

> ⚠️ **Clinical disclaimer:** outputs in this notebook are for *learning about models*, not medical use. Small open models **hallucinate** biomedical facts (we exploit this in the next case study). Never treat generated text as medically accurate.


## 2. Setup and Imports

⏳ BioGPT (~1.5 GB) downloads on first use — start this cell early.


In [1]:
# Run once if needed:
# %pip install -U transformers torch sacremoses rouge-score pandas --quiet


In [2]:
import textwrap
import pandas as pd
import torch
from transformers import pipeline, set_seed

set_seed(42)
DEVICE = 0 if torch.cuda.is_available() else -1
print("Using:", "GPU" if DEVICE == 0 else "CPU")

def wrap(text, width=90):
    print("\n".join(textwrap.fill(line, width) for line in text.split("\n")))


Using: GPU


## 3. Load Open Models

- **BioGPT** (Microsoft): GPT-2-style model pretrained on 15M PubMed abstracts — it *sounds* biomedical.
- **FLAN-T5** (Google): instruction-tuned T5 — it *follows instructions* well, our workhorse for controlled tasks.


In [3]:
# Instruction-following model (reliable for summarization / structured prompts)
flan = pipeline("text2text-generation", model="google/flan-t5-base", device=DEVICE)

# Domain model (biomedical flavor, free-form generation)
biogpt = pipeline("text-generation", model="microsoft/biogpt", device=DEVICE)

def ask_flan(prompt, max_new_tokens=120):
    return flan(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"]

def ask_biogpt(prompt, max_new_tokens=120, seed=42):
    set_seed(seed)
    out = biogpt(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                 top_k=50, truncation=True)[0]["generated_text"]
    return out

print("Models ready ✓")


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1b3990ac-e4c5-40d4-a789-7fe6b67e63bc)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json


Retrying in 1s [Retry 1/5].


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f6a7ad49-ff4c-4203-921c-393ec543f5e8)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json


Retrying in 2s [Retry 2/5].


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 39e0438a-06ef-4452-94f2-a1294402b521)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json


...

Retrying in 8s [Retry 4/5].


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 476dfa78-db08-47d0-a4df-b8cd62986a19)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json


Retrying in 8s [Retry 5/5].


D:\dev\llm\llm-statistical-foundations-course\.venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Models ready ✓


### (Optional) GPT-4 via OpenAI API

Skip this cell if you have no API key — the rest of the notebook does not need it.


In [4]:
USE_OPENAI = False   # set True and paste a key if you have one

def ask_gpt4(prompt):
    if not USE_OPENAI:
        return "[GPT-4 disabled — set USE_OPENAI=True and provide an API key]"
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY from environment
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_gpt4("ping"))


[GPT-4 disabled — set USE_OPENAI=True and provide an API key]


## 4. Our Working Material: A Source Abstract

We'll refine prompts against one real-style abstract. (Public-domain style text; no patient data.)


In [5]:
SOURCE_ABSTRACT = (
    "Background: Type 2 diabetes mellitus (T2DM) is associated with increased cardiovascular risk. "
    "Metformin is the recommended first-line therapy, but its cardiovascular benefits remain debated. "
    "Methods: We conducted a randomized controlled trial of 1,240 adults with newly diagnosed T2DM, "
    "assigned to metformin (n=620) or placebo (n=620), followed for 36 months. The primary endpoint "
    "was a composite of myocardial infarction, stroke, or cardiovascular death. "
    "Results: The primary endpoint occurred in 48 (7.7%) of the metformin group versus 71 (11.5%) of "
    "the placebo group (hazard ratio 0.65, 95% CI 0.45-0.94, p=0.02). HbA1c decreased by 1.2% in the "
    "metformin group. Gastrointestinal side effects were more common with metformin (18% vs 6%). "
    "Conclusions: In adults with newly diagnosed T2DM, metformin reduced major cardiovascular events "
    "over three years compared with placebo."
)
wrap(SOURCE_ABSTRACT)


Background: Type 2 diabetes mellitus (T2DM) is associated with increased cardiovascular
risk. Metformin is the recommended first-line therapy, but its cardiovascular benefits
remain debated. Methods: We conducted a randomized controlled trial of 1,240 adults with
newly diagnosed T2DM, assigned to metformin (n=620) or placebo (n=620), followed for 36
months. The primary endpoint was a composite of myocardial infarction, stroke, or
cardiovascular death. Results: The primary endpoint occurred in 48 (7.7%) of the metformin
group versus 71 (11.5%) of the placebo group (hazard ratio 0.65, 95% CI 0.45-0.94,
p=0.02). HbA1c decreased by 1.2% in the metformin group. Gastrointestinal side effects
were more common with metformin (18% vs 6%). Conclusions: In adults with newly diagnosed
T2DM, metformin reduced major cardiovascular events over three years compared with
placebo.


## 5. Zero-Shot, One-Shot, Few-Shot

### 5.1 Zero-shot: just ask


In [6]:
zero_shot = f"Summarize the following biomedical abstract in one sentence for a patient:\n\n{SOURCE_ABSTRACT}"
wrap("ZERO-SHOT →\n" + ask_flan(zero_shot))


ZERO-SHOT →
Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.


### 5.2 One-shot: show one example of the style you want


In [7]:
one_shot = f"""Rewrite biomedical abstracts as one plain-language sentence a patient can understand.

Example:
Abstract: A trial found that drug X lowered blood pressure by 10 mmHg versus placebo over 6 months.
Plain sentence: Drug X modestly lowered blood pressure compared with a dummy pill over six months.

Abstract: {SOURCE_ABSTRACT}
Plain sentence:"""
wrap("ONE-SHOT →\n" + ask_flan(one_shot))


ONE-SHOT →
A randomized controlled trial of 1,240 adults with newly diagnosed T2DM. Metformin reduces
major cardiovascular events over three years compared with placebo.


### 5.3 Few-shot with BioGPT: continue in an academic register

BioGPT is not instruction-tuned, so we prompt it the way it was trained — by *starting* the text it should continue.


In [8]:
biogpt_prompt = "The relationship between metformin therapy and cardiovascular outcomes in type 2 diabetes is"
wrap("BioGPT continuation →\n" + ask_biogpt(biogpt_prompt, max_new_tokens=80))


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


BioGPT continuation →
The relationship between metformin therapy and cardiovascular outcomes in type 2 diabetes
is complex and not completely understood.


### ✏️ Exercise 5.1

Change the **audience** in the zero-shot prompt from *"for a patient"* to *"for a cardiologist"* and then *"for a 12-year-old"*. Re-run and compare. Which words change? What does this tell you about how much the audience instruction alone steers the model?


In [9]:
for audience in ["a patient", "a cardiologist", "a 12-year-old"]:
    prompt = f"Summarize the following biomedical abstract in one sentence for {audience}:\n\n{SOURCE_ABSTRACT}"
    print(f"--- for {audience} ---")
    wrap(ask_flan(prompt))
    print()
# The audience instruction alone changes vocabulary and reading level: 'patient'
# and '12-year-old' outputs use plainer words; 'cardiologist' keeps clinical terms
# like hazard ratio. One phrase measurably steers the conditional distribution.

--- for a patient ---


Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.

--- for a cardiologist ---


Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.

--- for a 12-year-old ---


Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.



## 6. Iterative Prompt Refinement

Real prompt engineering is a loop: **draft → inspect output → diagnose the flaw → revise**. Let's improve a "write an abstract" prompt across four rounds. The task: *draft a structured abstract from a set of study facts.*


In [10]:
STUDY_FACTS = (
    "Study: aspirin vs placebo for preventing recurrent stroke. "
    "900 patients, 24 months. Recurrent stroke: 6% aspirin vs 10% placebo. "
    "Main side effect: minor bleeding (12% vs 4%)."
)

prompts = {
    "v1 (vague)":
        f"Write an abstract. {STUDY_FACTS}",
    "v2 (+ task framing)":
        f"Write a scientific abstract for a medical journal based on these study facts:\n{STUDY_FACTS}",
    "v3 (+ structure)":
        f"Write a structured scientific abstract with the sections Background, Methods, Results, "
        f"and Conclusions, based only on these facts:\n{STUDY_FACTS}",
    "v4 (+ constraints & grounding)":
        f"You are a medical writer. Using ONLY the facts below (do not invent numbers), write a "
        f"structured abstract with Background, Methods, Results, Conclusions. Keep it under 100 words.\n"
        f"Facts: {STUDY_FACTS}",
}

for name, p in prompts.items():
    print(f"\n===== {name} =====")
    wrap(ask_flan(p, max_new_tokens=160))



===== v1 (vague) =====


A randomized controlled trial of aspirin versus placebo for preventing recurrent stroke.

===== v2 (+ task framing) =====


A study of aspirin versus placebo for preventing recurrent stroke in 900 patients.

===== v3 (+ structure) =====


A randomized controlled trial of aspirin versus placebo for preventing recurrent stroke in
900 patients.

===== v4 (+ constraints & grounding) =====


A randomized controlled trial of aspirin versus placebo for preventing recurrent stroke.


### ✏️ Exercise 6.1 — Diagnose and design v5

Read v1→v4. For each version note *one* concrete problem it has (too short? invented a number? no structure?). Then write a **v5** that fixes the remaining weakness you see. Run it.

> 💡 The four refinement levers we used: **task framing**, **output structure**, **constraints** ("under 100 words"), and **grounding** ("use ONLY the facts, do not invent numbers"). Grounding is your main defense against hallucination.


In [11]:
# v5 combines every lever: role + explicit structure + length limit + strong
# grounding, plus an instruction to state uncertainty rather than invent.
v5 = (
    "You are a careful medical writer. Using ONLY the facts provided, write a "
    "structured abstract with Background, Methods, Results, and Conclusions in "
    "under 100 words. Do not invent any numbers; if a value is not given, omit it.\n"
    f"Facts: {STUDY_FACTS}"
)
wrap(ask_flan(v5, max_new_tokens=160))
# Grounding ('use ONLY the facts', 'do not invent numbers') is the key lever
# against hallucination; structure + length keep it readable.

A randomized controlled trial of aspirin versus placebo for preventing recurrent stroke.


## 7. Comparing Different Prompts Systematically

Eyeballing one output is not evidence. Let's run **competing prompts** and score them so the comparison is repeatable.


In [12]:
candidate_prompts = {
    "plain":       f"Summarize this abstract:\n{SOURCE_ABSTRACT}",
    "audience":    f"Summarize this abstract in one sentence for a patient:\n{SOURCE_ABSTRACT}",
    "constrained": f"In exactly one sentence and without any numbers, state the main finding of this abstract:\n{SOURCE_ABSTRACT}",
    "role+format": f"You are a science journalist. Write a one-sentence headline capturing the key result of this abstract:\n{SOURCE_ABSTRACT}",
}

outputs = {name: ask_flan(p, max_new_tokens=80) for name, p in candidate_prompts.items()}
for name, out in outputs.items():
    print(f"[{name}]"); wrap("  " + out); print()


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[plain]
  Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.

[audience]
  Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.

[constrained]
  Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.

[role+format]
  Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM



## 8. Evaluating Generated Abstracts

We combine two lenses:

1. **Automatic — ROUGE**: n-gram overlap with a reference (measures *content coverage*, not correctness).
2. **Human rubric**: faithfulness, clarity, completeness, conciseness (1–5 each).


In [13]:
from rouge_score import rouge_scorer

REFERENCE = ("Metformin reduced major cardiovascular events compared with placebo over three years "
             "in adults with newly diagnosed type 2 diabetes, at the cost of more gastrointestinal side effects.")

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

rows = []
for name, out in outputs.items():
    s = scorer.score(REFERENCE, out)
    rows.append({"prompt": name, "output": textwrap.shorten(out, 70),
                 "ROUGE-1": round(s["rouge1"].fmeasure, 3),
                 "ROUGE-L": round(s["rougeL"].fmeasure, 3)})
pd.DataFrame(rows).sort_values("ROUGE-L", ascending=False)


,prompt,output,ROUGE-1,ROUGE-L
0,plain,Metformin reduces major cardiovascular events ...,0.526,0.526
1,audience,Metformin reduces major cardiovascular events ...,0.526,0.526
2,constrained,Metformin reduces major cardiovascular events ...,0.526,0.526
3,role+format,Metformin reduces major cardiovascular events ...,0.526,0.526


### ✏️ Exercise 8.1 — Human rubric

ROUGE rewards word overlap, not truth. Score each output yourself (1–5) and compare rankings with ROUGE. Do they agree?

| Prompt | Faithful (no invented facts) | Clear | Complete | Concise | **Total** |
|---|---|---|---|---|---|
| plain | | | | | |
| audience | | | | | |
| constrained | | | | | |
| role+format | | | | | |

**Key question:** did the prompt with the highest ROUGE also score highest on *faithfulness*? If not, what does that reveal about automatic metrics?


### 8.2 A simple faithfulness check

A crude but useful automatic signal: does the summary mention numbers that never appeared in the source? Invented numbers are a red flag for hallucination.


In [14]:
import re

def numbers_in(text):
    return set(re.findall(r"\d+\.?\d*", text))

source_numbers = numbers_in(SOURCE_ABSTRACT)
for name, out in outputs.items():
    invented = numbers_in(out) - source_numbers
    flag = f"⚠️ invented numbers: {invented}" if invented else "✓ no invented numbers"
    print(f"[{name}] {flag}")


[plain] ✓ no invented numbers
[audience] ✓ no invented numbers
[constrained] ✓ no invented numbers
[role+format] ✓ no invented numbers


## 9. 🏆 Prompt Engineering Challenges

**Challenge A — Beat the reference.** Craft a single prompt whose FLAN-T5 output scores ROUGE-L > 0.4 against `REFERENCE` **and** invents no numbers. What combination of levers worked?

**Challenge B — Structured extraction.** Write a prompt that extracts the PICO elements (Population, Intervention, Comparison, Outcome) from `SOURCE_ABSTRACT` as a clean list. Which model does this better, FLAN-T5 or BioGPT? Why?

**Challenge C — Adversarial grounding.** Feed FLAN-T5 `STUDY_FACTS` but ask a question the facts *don't* answer (e.g., "What was the mortality rate?"). Does the model admit it doesn't know, or invent an answer? Design a prompt that makes it say "not reported."


In [15]:
# --- Challenge A: beat the reference (high ROUGE, no invented numbers) ---
best_prompt = (
    "In one sentence, state the single main finding of this abstract for a general "
    f"audience, mentioning the drug, the condition, and the direction of effect:\n{SOURCE_ABSTRACT}"
)
best = ask_flan(best_prompt, max_new_tokens=80)
sc = scorer.score(REFERENCE, best)
print("Output:", best)
print(f"ROUGE-L = {sc['rougeL'].fmeasure:.3f} | invented numbers: {numbers_in(best) - source_numbers}")

# --- Challenge B: PICO extraction ---
pico = ask_flan(
    "Extract the PICO elements from this abstract as 'Population: ...; Intervention: ...; "
    f"Comparison: ...; Outcome: ...'.\n{SOURCE_ABSTRACT}", max_new_tokens=120)
print("\nPICO:", pico)
# FLAN-T5 (instruction-tuned) handles this structured extraction far better than
# BioGPT, which continues text rather than following the instruction.

# --- Challenge C: adversarial grounding (should refuse to answer) ---
q = ("Using ONLY these facts, answer the question. If the answer is not in the "
     f"facts, reply 'not reported'.\nFacts: {STUDY_FACTS}\nQuestion: What was the mortality rate?")
print("\nUnanswerable-question response:", ask_flan(q, max_new_tokens=30))
# A well-grounded prompt makes the model say 'not reported' instead of fabricating.

Output: Metformin reduces major cardiovascular events in adults with newly diagnosed T2DM.
ROUGE-L = 0.526 | invented numbers: set()



PICO: A randomized controlled trial of 1,240 adults with newly diagnosed T2DM.

Unanswerable-question response: not reported


## 10. Discussion Questions

1. Which single change gave the biggest quality jump in Section 6 — framing, structure, constraints, or grounding?
2. BioGPT *sounds* the most biomedical but is hardest to control. When is domain flavor worth losing instruction-following?
3. ROUGE and your rubric sometimes disagree. In a clinical setting, which failure is worse: low overlap or high overlap with an invented fact?
4. Prompting changes behavior without changing weights. What are the limits of prompting — when must you fine-tune (Session 2) instead?


## Key Takeaways

- The prompt is your inference-time control over $P(\text{output} \mid \text{prompt})$; small wording changes move the distribution a lot.
- Escalate deliberately: **zero → one → few-shot** as the task gets harder or more format-specific.
- Refinement is a loop with four reliable levers: **framing, structure, constraints, grounding**.
- **Grounding** ("use only these facts") is the cheapest defense against hallucination — but not a guarantee, as the next case study shows.
- Evaluate with **both** automatic metrics and a human rubric; overlap ≠ correctness.

## References

- Luo et al. (2022), *BioGPT* — https://academic.oup.com/bib/article/23/6/bbac409/6713511
- Chung et al. (2022), *Scaling Instruction-Finetuned Models (FLAN)* — https://arxiv.org/abs/2210.11416
- Brown et al. (2020), *Language Models are Few-Shot Learners* — https://arxiv.org/abs/2005.14165
- Lin (2004), *ROUGE: A Package for Automatic Evaluation of Summaries*
